# Dafne Thigh Segmentation — Sheffield — Retry (Aug_50, Aug_65)

Runs Dafne on **only Aug_50 and Aug_65** — the two volumes that produced empty
output directories in the full run.

Data: `~/sheffeld/20440164/Aug_N.dcm`  
Output: `~/dafne_sheffield_segs/Aug_N/Aug_N_dafne_thigh.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/model/" \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_model/
```

## 2 — Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  --include='Aug_50/' --include='Aug_50/**' \
  --include='Aug_65/' --include='Aug_65/**' \
  --exclude='*' \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'dafne-dl', 'SimpleITK', 'pydicom'])
print('Dependencies installed — restart the kernel now, then run remaining cells.')

In [ ]:
import glob, os, re
import numpy as np
import pydicom
from dafne_dl import DynamicDLModel

IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
OUTPUT_DIR = os.path.expanduser('~/dafne_sheffield_segs')

_model_candidates = sorted(glob.glob(os.path.expanduser('~/dafne_model/*.model')))
if not _model_candidates:
    raise FileNotFoundError('No .model file found in ~/dafne_model/ — upload it first')
MODEL_PATH = _model_candidates[0]

# Only the two volumes that failed in the full run
RETRY_INDICES = [50, 65]

dcm_files = []
for idx in RETRY_INDICES:
    p = os.path.join(IMG_DIR, f'Aug_{idx}.dcm')
    if os.path.exists(p):
        dcm_files.append(p)
    else:
        print(f'WARNING: {p} not found')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model : {MODEL_PATH}')
print(f'Volumes to retry: {[os.path.basename(p) for p in dcm_files]}')

In [ ]:
model = DynamicDLModel.Load(open(MODEL_PATH, 'rb'))
print('Model loaded.')

In [ ]:
def read_dicom_volume(dcm_path):
    ds  = pydicom.dcmread(dcm_path)
    arr = ds.pixel_array.astype(np.float32)
    ps  = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    return arr, [float(ps[0]), float(ps[1])]

# Lower threshold than the original (0.01) in case these volumes have weak signal
EMPTY_THRESHOLD = 0.005

for dcm_path in dcm_files:
    idx = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    out_subdir = os.path.join(OUTPUT_DIR, f'Aug_{idx}')
    out_path   = os.path.join(out_subdir, f'Aug_{idx}_dafne_thigh.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\nProcessing: Aug_{idx}')
    os.makedirs(out_subdir, exist_ok=True)

    img_array, resolution = read_dicom_volume(dcm_path)
    D, H, W = img_array.shape
    print(f'  Shape: {img_array.shape}  Resolution: {resolution}')
    print(f'  Pixel range: [{img_array.min():.1f}, {img_array.max():.1f}]')

    v_min, v_max = float(img_array.min()), float(img_array.max())
    img_norm = (img_array - v_min) / (v_max - v_min + 1e-8)

    all_masks = {}
    skipped = 0
    for sl in range(D):
        slice_img = img_norm[sl]

        if float(slice_img.max()) < EMPTY_THRESHOLD:
            skipped += 1
            continue

        try:
            out = model({
                'image':            slice_img,
                'resolution':       resolution,
                'split_laterality': True,
                'classification':   'Thigh',
            })
        except RuntimeError as e:
            print(f'  [skip slice {sl}] {e}')
            skipped += 1
            continue

        for name, mask in out.items():
            if name not in all_masks:
                all_masks[name] = np.zeros((D, H, W), dtype=np.uint8)
            all_masks[name][sl] = np.asarray(mask, dtype=np.uint8)

        if (sl + 1) % 50 == 0 or sl == D - 1:
            print(f'  slice {sl+1}/{D}  (skipped so far: {skipped})')

    if all_masks:
        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved → {out_path}  muscles: {list(all_masks.keys())}  skipped: {skipped}/{D}')
    else:
        print(f'  WARNING: no masks produced for Aug_{idx} — all {D} slices skipped')
        print(f'  (pixel max after norm = {img_norm.max():.4f}, EMPTY_THRESHOLD = {EMPTY_THRESHOLD})')

print('\nDone.')

In [ ]:
for idx in RETRY_INDICES:
    p = os.path.join(OUTPUT_DIR, f'Aug_{idx}', f'Aug_{idx}_dafne_thigh.npz')
    if os.path.exists(p):
        s = np.load(p)
        print(f'Aug_{idx}: {list(s.files)}')
        for k in s.files:
            print(f'  {k}: voxels={int(s[k].sum()):,}')
    else:
        print(f'Aug_{idx}: MISSING — segmentation failed')